**Installing Packages**

In [1]:
!pip install streamlit langchain langchain-anthropic
!pip install pandas matplotlib python-dotenv
!pip install duckduckgo-search pypdf faiss-cpu pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 59.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 106.6 MB/s eta 0:00:00


**Writing the app**

In [2]:
%%writefile app.py

import streamlit as st
from langchain_anthropic import ChatAnthropic
from langchain.tools import DuckDuckGoSearchRun
from langchain.agents import initialize_agent
from langchain.prompts import PromptTemplate
import pandas as pd


import os
os.environ["ANTHROPIC_API_KEY"] = "your_api_key_here"

st.set_page_config(page_title="All-in-One AI Agent", page_icon="🤖")
st.title("🤖 All-in-One AI Agent")

agent_type = st.sidebar.selectbox("Choose Your Agent", [
    "📊 Data Analyst",
    "🔍 Research Agent",
    "📧 Email Agent",
])

# Data Analyst
if agent_type == "📊 Data Analyst":
    st.header("📊 Data Analyst Agent")
    file = st.file_uploader("Upload CSV", type="csv")
    question = st.text_input("Ask anything about your data")
    if st.button("Analyze") and file and question:
        df = pd.read_csv(file)
        st.dataframe(df.head())
        llm = ChatAnthropic(model="claude-sonnet-4-6")
        from langchain.agents import create_pandas_dataframe_agent
        agent = create_pandas_dataframe_agent(llm, df, verbose=True)
        result = agent.run(question)
        st.success(result)

# Research Agent
elif agent_type == "🔍 Research Agent":
    st.header("🔍 Research Agent")
    query = st.text_input("What do you want to research?")
    if st.button("Search") and query:
        llm = ChatAnthropic(model="claude-sonnet-4-6")
        search = DuckDuckGoSearchRun()
        agent = initialize_agent(
            tools=[search], llm=llm,
            agent="zero-shot-react-description"
        )
        result = agent.run(query)
        st.write(result)

# Email Agent
elif agent_type == "📧 Email Agent":
    st.header("📧 Email Agent")
    context = st.text_area("What is the email about?")
    tone = st.selectbox("Tone", ["Professional", "Casual", "Formal"])
    if st.button("Write Email") and context:
        llm = ChatAnthropic(model="claude-sonnet-4-6")
        prompt = PromptTemplate.from_template(
            "Write a {tone} email about: {context}"
        )
        chain = prompt | llm
        result = chain.invoke({"context": context, "tone": tone})
        st.write(result.content)

Writing app.py


**Run with ngrok**

In [6]:
from pyngrok import conf
ngrok.set_auth_token("3JKTpI5firoIgGjUPgIkL4lkEoW_7JGeAPfFJ7tf5x5MbBHWq")

In [7]:
from pyngrok import ngrok
import subprocess
import time

# Set your auth token FIRST
ngrok.set_auth_token("3JKTpI5firoIgGjUPgIkL4lkEoW_7JGeAPfFJ7tf5x5MbBHWq")

# Start Streamlit in background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"]
)

time.sleep(5)

# Create public URL
public_url = ngrok.connect(8501)
print(f"✅ Your app is live at: {public_url}")

✅ Your app is live at: NgrokTunnel: "https://paternal-sweat-cymbal.ngrok-free.dev" -> "http://localhost:8501"
